# CS-781 Capstone Project: Phase 4 - Court Judgment Prediction & Explanation (CJPE)
## Submission 07: Model 4 Peak Capacity Boundary (154M) with Hierarchical Cross-Chunk Attention Pooling

### Executive Architectural Summary
This notebook implements **Hierarchical Cross-Chunk Transformer Attention Pooling** for binary Court Judgment Prediction.
- **Base Pretrained Backbone**: Scratch-trained Phase 1 `model4-capacity-boundary-154m` ($14$ layers, $d_{\text{model}}=896$, Tied Embeddings, $153.9\text{M}$ parameters).
- **Hierarchical Cross-Chunk Attention Mechanism**:
  - Long judgment texts (often spanning thousands of tokens) are segmented into sliding chunks of $512$ tokens with a $256$-token stride ($K=4$ chunks).
  - The frozen or fine-tuned backbone produces token-level contextual representations for each chunk, pooled into chunk embedding vectors $e_k \in \mathbb{R}^{896}$.
  - A 2-layer Cross-Chunk Multi-Head Attention Head ($n_{\text{head}}=8$, $d_{\text{ff}}=1792$) contextualizes relationships across distinct judicial sections (facts, issues, legal precedents, rationale).
  - Learned attentive pooling weights $w_k = \text{Softmax}(W_c \cdot h_k)$ compress chunk embeddings into a single unified judgment document vector for binary classification.
- **Loss Objective**: Binary Cross-Entropy with Logits and Label Smoothing ($\epsilon=0.05$).
- **Threshold Tuning**: Development split threshold sweep $\theta \in [0.35, 0.65]$ maximizing Macro-F1 on `dev.json`.
- **Cloud Infrastructure**: Continuous background Hugging Face Hub checkpoint pushing every 200 steps (or 20 steps in SMOKE mode), crash recovery, and automated CodaBench zip formatting.


In [ ]:
# Cell 1: Dual-GPU Detection & Profile Configuration
import os
import sys
import time
import torch

print("=" * 80)
print("CS-781 Phase 4: Sub-07 Model 4 Hierarchical Cross-Chunk Attention Binary CJPE")
print("=" * 80)

# Execution Profile Toggle: 'SMOKE' for rapid 1-minute test, 'CORE' for production
PROFILE = "CORE"  # Change to "SMOKE" for ultra-fast pre-flight dry-run
print(f"[CONFIG] Active Execution Profile: {PROFILE}")

# GPU Detection
num_gpus = torch.cuda.device_count()
print(f"[HARDWARE] Available CUDA GPUs: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | VRAM: {torch.cuda.get_device_properties(i).total_memory / (1024**3):.2f} GB")

if num_gpus == 0:
    print("[WARNING] No GPU detected! Running on CPU fallback.")
    device = torch.device("cpu")
else:
    device = torch.device("cuda:0")
    print(f"[HARDWARE] Primary compute device: {device}")
    if hasattr(torch.backends, "cuda") and hasattr(torch.backends.cuda, "matmul"):
        torch.backends.cuda.matmul.allow_tf32 = True

CONFIG = {
    "SMOKE": {
        "max_train_samples": 50,
        "max_dev_samples": 25,
        "max_test_samples": 25,
        "batch_size": 2,
        "gradient_accumulation_steps": 4,
        "chunk_size": 256,
        "chunk_stride": 128,
        "max_chunks": 3,
        "max_steps": 30,
        "eval_interval": 10,
        "save_interval": 20,
        "learning_rate": 2e-5,
        "warmup_steps": 5,
        "label_smoothing": 0.05,
    },
    "CORE": {
        "max_train_samples": None,
        "max_dev_samples": None,
        "max_test_samples": None,
        "batch_size": 4,
        "gradient_accumulation_steps": 4,
        "chunk_size": 512,
        "chunk_stride": 256,
        "max_chunks": 4,
        "max_steps": 2500,
        "eval_interval": 200,
        "save_interval": 200,
        "learning_rate": 2e-5,
        "warmup_steps": 250,
        "label_smoothing": 0.05,
    }
}

CFG = CONFIG[PROFILE]
print("[CONFIG] Loaded parameters:")
for k, v in CFG.items():
    print(f"  {k}: {v}")


In [ ]:
# Cell 2: Dependencies Installation & Environment Setup
import subprocess

packages = [
    "transformers>=4.38.0",
    "accelerate>=0.27.0",
    "scikit-learn>=1.3.0",
    "huggingface_hub>=0.20.0"
]

print("[INSTALL] Verifying and installing required packages...")
for pkg in packages:
    pkg_name = pkg.split(">=")[0].replace("-", "_")
    try:
        __import__(pkg_name)
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("[INSTALL] Dependencies successfully verified.")


In [ ]:
# Cell 3: Secrets & Hugging Face Hub Authentication
from pathlib import Path
from huggingface_hub import HfApi, login

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    print("[AUTH] Successfully loaded HF_TOKEN from Kaggle Secrets.")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if HF_TOKEN:
        print("[AUTH] Loaded HF_TOKEN from environment variable.")
    else:
        print("[AUTH] Note: HF_TOKEN not found. Remote sync will operate in dry-run mode.")

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=True)
    api = HfApi(token=HF_TOKEN)
    try:
        user_info = api.whoami()
        print(f"[AUTH] Authenticated as Hugging Face user: {user_info['name']}")
    except Exception as e:
        print(f"[AUTH-WARNING] Verification warning: {e}")
else:
    api = None

HUB_REPO_ID = "sarweshbarnwal/CS781-Legal-Models"
HUB_SUBFOLDER = "Phase_4/sub07_model4_hierarchical_pool"
LOCAL_DIR = Path("./phase4_sub07_hierarchical_run")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CHECKPOINT_DIR = LOCAL_DIR / "checkpoints"
LOCAL_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT_DIR = LOCAL_DIR / "outputs"
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_SUBMISSION_DIR = LOCAL_DIR / "submission"
LOCAL_SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

print(f"[CONFIG] Checkpoint Target: {HUB_REPO_ID}/{HUB_SUBFOLDER}")


In [ ]:
# Cell 4: CJPE Dataset Ingestion & Pretrained Checkpoint Loading
import json
import shutil
from huggingface_hub import hf_hub_download, snapshot_download

DATA_DIR = Path("./phase4_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

train_path = DATA_DIR / "train.json"
dev_path = DATA_DIR / "dev.json"
test_path = DATA_DIR / "test.json"

print("[DATA] Ingesting CJPE dataset from Exploration-Lab/CS-781-Capstone...")
try:
    if not train_path.exists():
        print("Downloading train.json from Exploration-Lab/CS-781-Capstone...")
        downloaded = hf_hub_download(
            repo_id="Exploration-Lab/CS-781-Capstone",
            subfolder="Phase_4_CJPE/Train",
            filename="train.json",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
            local_dir_use_symlinks=False
        )
        if Path(downloaded) != train_path and Path(downloaded).exists():
            shutil.copy(downloaded, train_path)
    if not dev_path.exists():
        print("Downloading dev.json from Exploration-Lab/CS-781-Capstone...")
        downloaded = hf_hub_download(
            repo_id="Exploration-Lab/CS-781-Capstone",
            subfolder="Phase_4_CJPE/Train",
            filename="dev.json",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
            local_dir_use_symlinks=False
        )
        if Path(downloaded) != dev_path and Path(downloaded).exists():
            shutil.copy(downloaded, dev_path)
    if not test_path.exists():
        print("Downloading test.json from Exploration-Lab/CS-781-Capstone...")
        downloaded = hf_hub_download(
            repo_id="Exploration-Lab/CS-781-Capstone",
            subfolder="Phase_4_CJPE/Test",
            filename="test.json",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
            local_dir_use_symlinks=False
        )
        if Path(downloaded) != test_path and Path(downloaded).exists():
            shutil.copy(downloaded, test_path)
except Exception as e:
    print(f"[DATA] Hub download note ({e}), checking fallback...")

if not train_path.exists() or os.path.getsize(train_path) < 100:
    print("[DATA] Creating high-fidelity fallback dataset for offline/smoke execution...")
    demo_train = [
        {
            "id": f"case_tr_{i:04d}",
            "text": f"Criminal Appeal No. {100+i}. Prosecution under Sections 302, 34 of IPC. Eye-witness testimony corroborated with ballistic medical reports. Accused convicted.",
            "label": i % 2
        }
        for i in range(100)
    ]
    with open(train_path, "w", encoding="utf-8") as f:
        json.dump(demo_train, f, indent=2)

if not dev_path.exists() or os.path.getsize(dev_path) < 100:
    demo_dev = [
        {
            "id": f"case_dev_{i:04d}",
            "text": f"Civil Appeal No. {200+i}. Specific Relief Act and Property disputes under relevant legal statutes. Title determination and evidence scrutiny.",
            "label": i % 2
        }
        for i in range(50)
    ]
    with open(dev_path, "w", encoding="utf-8") as f:
        json.dump(demo_dev, f, indent=2)

if not test_path.exists() or os.path.getsize(test_path) < 100:
    demo_test = [
        {
            "id": f"case_te_{i:04d}",
            "text": f"Special Leave Petition No. {300+i}. Question of statutory interpretation of Code of Criminal Procedure and Indian Penal Code.",
        }
        for i in range(30)
    ]
    with open(test_path, "w", encoding="utf-8") as f:
        json.dump(demo_test, f, indent=2)

with open(train_path, "r", encoding="utf-8") as f:
    raw_train_data = json.load(f)
with open(dev_path, "r", encoding="utf-8") as f:
    raw_dev_data = json.load(f)
with open(test_path, "r", encoding="utf-8") as f:
    raw_test_data = json.load(f)

print(f"[DATA] Ingested {len(raw_train_data)} train, {len(raw_dev_data)} dev, and {len(raw_test_data)} test documents.")

# Download Model 4 Pretrained Checkpoint
print("[MODEL] Downloading Phase 1 Model 4 Boundary Decoder (154M)...")
try:
    snapshot_download(
        repo_id="sarweshbarnwal/CS781-Legal-Models",
        allow_patterns=["model4-capacity-boundary-154m/*"],
        local_dir="./model4_snapshot"
    )
    ckpt_folder = Path("./model4_snapshot/model4-capacity-boundary-154m")
    if ckpt_folder.exists():
        MODEL_PATH = str(ckpt_folder)
    else:
        MODEL_PATH = "sarweshbarnwal/CS781-Legal-Models"
except Exception as e:
    print(f"[MODEL] Direct snapshot download warning: {e}. Using repo subfolder reference.")
    MODEL_PATH = "sarweshbarnwal/CS781-Legal-Models"

print(f"[MODEL] Foundation model path resolved to: {MODEL_PATH}")


In [ ]:
# Cell 5: Label-Smoothed Binary Cross-Entropy Loss Implementation
import torch
import torch.nn as nn
import torch.nn.functional as F

class LabelSmoothedBCEWithLogitsLoss(nn.Module):
    def __init__(self, eps: float = 0.05):
        super().__init__()
        self.eps = eps

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # Smooth targets: y_smooth = y * (1 - eps) + 0.5 * eps
        targets_smooth = targets * (1.0 - self.eps) + 0.5 * self.eps
        return F.binary_cross_entropy_with_logits(logits, targets_smooth)

print("[ARCH] Label Smoothed BCE Loss module compiled successfully.")


In [ ]:
# Cell 6: Sliding Multi-Chunk Dataset Tokenization & Preparation
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# Load Tokenizer
tokenizer_source = MODEL_PATH if Path(MODEL_PATH).exists() else "sarweshbarnwal/CS781-Legal-Models"
try:
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_source, subfolder="model4-capacity-boundary-154m" if "/" in tokenizer_source and not Path(tokenizer_source).exists() else None)
except Exception:
    print("[TOKENIZER] Loading fallback LLaMA-compatible tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained("hf-internal-testing/llama-tokenizer")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token is not None else "<pad>"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 3

print(f"[TOKENIZER] Vocab size: {len(tokenizer)}, Pad ID: {tokenizer.pad_token_id}")

class HierarchicalCJPEChunkDataset(Dataset):
    """
    Splits long legal judgments into sliding chunks:
    - chunk_size = 512, chunk_stride = 256, max_chunks = 4
    """
    def __init__(self, records, tokenizer, chunk_size=512, chunk_stride=256, max_chunks=4, is_test=False):
        self.samples = []
        for r in records:
            doc_id = str(r.get("id", ""))
            text = str(r.get("text", ""))
            label = float(r.get("label", 0)) if not is_test else 0.0

            self.samples.append({
                "id": doc_id,
                "text": text,
                "label": torch.tensor(label, dtype=torch.float32),
                "is_test": is_test
            })

        self.tokenizer = tokenizer
        self.chunk_size = chunk_size
        self.chunk_stride = chunk_stride
        self.max_chunks = max_chunks

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        tokens = self.tokenizer.encode(item["text"], add_special_tokens=True)

        chunk_ids = []
        chunk_masks = []

        start = 0
        for _ in range(self.max_chunks):
            if start >= len(tokens):
                break
            c_toks = tokens[start:start + self.chunk_size]
            pad_len = self.chunk_size - len(c_toks)
            padded = c_toks + [self.tokenizer.pad_token_id] * pad_len
            mask = [1] * len(c_toks) + [0] * pad_len

            chunk_ids.append(padded)
            chunk_masks.append(mask)
            start += self.chunk_stride

        active_chunks = len(chunk_ids)
        if active_chunks == 0:
            chunk_ids.append([self.tokenizer.pad_token_id] * self.chunk_size)
            chunk_masks.append([0] * self.chunk_size)
            active_chunks = 1

        while len(chunk_ids) < self.max_chunks:
            chunk_ids.append([self.tokenizer.pad_token_id] * self.chunk_size)
            chunk_masks.append([0] * self.chunk_size)

        c_mask = [1.0] * active_chunks + [0.0] * (self.max_chunks - active_chunks)

        return {
            "id": item["id"],
            "chunk_ids": torch.tensor(chunk_ids, dtype=torch.long),         # [max_chunks, chunk_size]
            "chunk_masks": torch.tensor(chunk_masks, dtype=torch.long),     # [max_chunks, chunk_size]
            "doc_chunk_mask": torch.tensor(c_mask, dtype=torch.float32),    # [max_chunks]
            "label": item["label"]
        }


def chunk_collate_fn(batch):
    return {
        "ids": [b["id"] for b in batch],
        "chunk_ids": torch.stack([b["chunk_ids"] for b in batch]),
        "chunk_masks": torch.stack([b["chunk_masks"] for b in batch]),
        "doc_chunk_mask": torch.stack([b["doc_chunk_mask"] for b in batch]),
        "labels": torch.stack([b["label"] for b in batch])
    }

train_records = raw_train_data
dev_records = raw_dev_data
test_records = raw_test_data

if CFG["max_train_samples"]:
    train_records = train_records[:CFG["max_train_samples"]]
if CFG["max_dev_samples"]:
    dev_records = dev_records[:CFG["max_dev_samples"]]
if CFG["max_test_samples"]:
    test_records = test_records[:CFG["max_test_samples"]]

train_dataset = HierarchicalCJPEChunkDataset(train_records, tokenizer, CFG["chunk_size"], CFG["chunk_stride"], CFG["max_chunks"])
dev_dataset = HierarchicalCJPEChunkDataset(dev_records, tokenizer, CFG["chunk_size"], CFG["chunk_stride"], CFG["max_chunks"])
test_dataset = HierarchicalCJPEChunkDataset(test_records, tokenizer, CFG["chunk_size"], CFG["chunk_stride"], CFG["max_chunks"], is_test=True)

print(f"[DATA] Prepared {len(train_dataset)} train, {len(dev_dataset)} dev, and {len(test_dataset)} test samples.")


In [ ]:
# Cell 7: Hierarchical Cross-Chunk Attention Pooling Architecture
from transformers import AutoModel, AutoConfig

class CrossChunkAttentionPooler(nn.Module):
    """
    2-Layer Cross-Chunk Multi-Head Attention Head over section chunks:
    Attends across all active document chunks to aggregate global legal context.
    """
    def __init__(self, hidden_size: int = 896, num_heads: int = 8, num_layers: int = 2):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=num_heads,
            dim_feedforward=hidden_size * 2,
            dropout=0.1,
            activation="gelu",
            batch_first=True
        )
        self.cross_chunk_transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.chunk_score = nn.Linear(hidden_size, 1)
        self.norm = nn.LayerNorm(hidden_size)
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, chunk_embeds: torch.Tensor, doc_chunk_mask: torch.Tensor) -> torch.Tensor:
        # chunk_embeds: [B, K, D], doc_chunk_mask: [B, K]
        padding_mask = (doc_chunk_mask == 0)

        # Contextualize chunks
        context_chunks = self.cross_chunk_transformer(chunk_embeds, src_key_padding_mask=padding_mask)

        # Attention pooling across chunks
        scores = self.chunk_score(context_chunks).squeeze(-1)  # [B, K]
        scores = scores.masked_fill(padding_mask, -1e9)
        weights = F.softmax(scores, dim=-1).unsqueeze(-1)      # [B, K, 1]

        pooled_doc = torch.sum(context_chunks * weights, dim=1) # [B, D]
        pooled_doc = self.norm(pooled_doc)
        logits = self.classifier(pooled_doc).squeeze(-1)       # [B]
        return logits, pooled_doc


class HierarchicalModel4CJPE(nn.Module):
    def __init__(self, backbone: nn.Module, hidden_size: int = 896):
        super().__init__()
        self.backbone = backbone
        self.pooler = CrossChunkAttentionPooler(hidden_size=hidden_size, num_heads=8, num_layers=2)

    def forward(self, chunk_ids: torch.Tensor, chunk_masks: torch.Tensor, doc_chunk_mask: torch.Tensor):
        # chunk_ids: [B, K, L], chunk_masks: [B, K, L]
        b_size, k_chunks, l_seq = chunk_ids.shape
        flat_ids = chunk_ids.view(b_size * k_chunks, l_seq)
        flat_mask = chunk_masks.view(b_size * k_chunks, l_seq)

        outputs = self.backbone(input_ids=flat_ids, attention_mask=flat_mask)
        hidden_states = outputs[0] if isinstance(outputs, tuple) else outputs.last_hidden_state

        mask_exp = flat_mask.unsqueeze(-1).expand_as(hidden_states).float()
        sum_hidden = torch.sum(hidden_states * mask_exp, dim=1)
        sum_mask = torch.clamp(mask_exp.sum(dim=1), min=1e-9)
        chunk_embeddings = (sum_hidden / sum_mask).view(b_size, k_chunks, -1)

        logits, pooled_doc = self.pooler(chunk_embeddings, doc_chunk_mask)
        return logits, pooled_doc

print("[MODEL] Loading base Model 4 Boundary Decoder backbone...")
try:
    if Path(MODEL_PATH).exists():
        base_backbone = AutoModel.from_pretrained(MODEL_PATH, torch_dtype=torch.float32)
    else:
        base_backbone = AutoModel.from_pretrained("sarweshbarnwal/CS781-Legal-Models", subfolder="model4-capacity-boundary-154m", torch_dtype=torch.float32)
except Exception:
    cfg = AutoConfig.for_model("llama", vocab_size=len(tokenizer), hidden_size=896, intermediate_size=2432, num_hidden_layers=14, num_attention_heads=14, num_key_value_heads=14)
    base_backbone = AutoModel.from_config(cfg)

model = HierarchicalModel4CJPE(base_backbone, hidden_size=base_backbone.config.hidden_size)
total_params = sum(p.numel() for p in model.parameters())
trainable_params_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"[MODEL] Hierarchical Model 4 instantiated: {total_params:,} total parameters ({trainable_params_count:,} trainable).")
model.to(device)
loss_fn = LabelSmoothedBCEWithLogitsLoss(eps=CFG["label_smoothing"])


In [ ]:
# Cell 8: DataLoader Setup & Mandatory 1-Batch Pre-Flight Gradient Verification
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    collate_fn=chunk_collate_fn,
    drop_last=(len(train_dataset) >= CFG["batch_size"]),
    num_workers=0
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    collate_fn=chunk_collate_fn,
    drop_last=False,
    num_workers=0
)

print("[PRE-FLIGHT] Verifying Hierarchical Cross-Chunk Attention forward-backward pass...")
model.train()
test_batch = next(iter(train_loader))
t_chunk_ids = test_batch["chunk_ids"].to(device)
t_chunk_masks = test_batch["chunk_masks"].to(device)
t_doc_mask = test_batch["doc_chunk_mask"].to(device)
t_labels = test_batch["labels"].to(device)

t_logits, _ = model(t_chunk_ids, t_chunk_masks, t_doc_mask)
t_loss = loss_fn(t_logits, t_labels)
print(f"[PRE-FLIGHT] Initial Label-Smoothed BCE Loss: {t_loss.item():.4f}")

t_loss.backward()

grad_bb_ok = any(p.grad is not None and p.grad.abs().sum() > 0 for p in model.backbone.parameters())
grad_pooler_ok = any(p.grad is not None and p.grad.abs().sum() > 0 for p in model.pooler.parameters())

print(f"[PRE-FLIGHT] Backbone Gradients Active: {grad_bb_ok}")
print(f"[PRE-FLIGHT] Cross-Chunk Attention Head Gradients Active: {grad_pooler_ok}")

if not (grad_bb_ok and grad_pooler_ok):
    raise RuntimeError("[ERROR] Hierarchical pipeline failed gradient check!")

for p in model.parameters():
    if p.grad is not None:
        p.grad.detach_()
        p.grad.zero_()

print("[PRE-FLIGHT] Verification PASSED! Hierarchical pipeline operational.")


In [ ]:
# Cell 9: Non-blocking Hugging Face Hub Continuous Sync Callback
import threading
from huggingface_hub import HfApi

class HfContinuousSyncCallback:
    def __init__(self, repo_id: str, subfolder: str, token: str = None):
        self.repo_id = repo_id
        self.subfolder = subfolder
        self.token = token
        self.api = HfApi(token=token) if token else None
        self.active_threads = []

    def _sync_worker(self, local_path: str, remote_path: str):
        if not self.api:
            print(f"[SYNC] Offline mode: Skipping cloud upload of {local_path}")
            return
        try:
            if os.path.isdir(local_path):
                self.api.upload_folder(
                    folder_path=local_path,
                    repo_id=self.repo_id,
                    path_in_repo=remote_path,
                    commit_message=f"Upload {remote_path} from Kaggle Phase 4 Sub-07 Hierarchical"
                )
            else:
                self.api.upload_file(
                    path_or_fileobj=local_path,
                    repo_id=self.repo_id,
                    path_in_repo=remote_path,
                    commit_message=f"Upload {remote_path} from Kaggle Phase 4 Sub-07 Hierarchical"
                )
            print(f"[SYNC] Successfully pushed to hub: {remote_path}")
        except Exception as e:
            print(f"[SYNC-WARNING] Hub upload error for {remote_path}: {e}")

    def push_checkpoint_async(self, checkpoint_dir: Path, step: int):
        remote_path = f"{self.subfolder}/checkpoint-{step}"
        t = threading.Thread(target=self._sync_worker, args=(str(checkpoint_dir), remote_path), daemon=True)
        t.start()
        self.active_threads.append(t)

    def wait_all(self):
        for t in self.active_threads:
            if t.is_alive():
                t.join(timeout=30)

hub_sync = HfContinuousSyncCallback(HUB_REPO_ID, HUB_SUBFOLDER, HF_TOKEN)
print("[SYNC] Cloud sync callback ready.")


In [ ]:
# Cell 10: Training Loop with Macro-F1 Validation & Threshold Calibration
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=CFG["learning_rate"], weight_decay=0.01)
lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=CFG["warmup_steps"],
    num_training_steps=CFG["max_steps"]
)

def evaluate_dev(eval_model, dataloader):
    eval_model.eval()
    all_logits = []
    all_targets = []
    with torch.no_grad():
        for batch in dataloader:
            chunk_ids = batch["chunk_ids"].to(device)
            chunk_masks = batch["chunk_masks"].to(device)
            doc_mask = batch["doc_chunk_mask"].to(device)
            labels = batch["labels"].cpu().numpy()

            logits, _ = eval_model(chunk_ids, chunk_masks, doc_mask)
            all_logits.append(logits.cpu().numpy())
            all_targets.append(labels)

    eval_model.train()
    all_logits = np.concatenate(all_logits, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    probs = 1.0 / (1.0 + np.exp(-all_logits))

    best_macro_f1 = 0.0
    best_th = 0.5
    for th in np.arange(0.35, 0.65, 0.02):
        preds = (probs >= th).astype(int)
        macro_f1 = f1_score(all_targets, preds, average="macro", zero_division=0)
        if macro_f1 > best_macro_f1:
            best_macro_f1 = macro_f1
            best_th = th

    preds_opt = (probs >= best_th).astype(int)
    acc = accuracy_score(all_targets, preds_opt)
    return best_macro_f1, acc, best_th

def save_hierarchical_checkpoint(save_dir: Path, step: int, threshold: float):
    save_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), save_dir / "hierarchical_model.pt")
    tokenizer.save_pretrained(save_dir)
    meta = {
        "step": step,
        "best_threshold": threshold,
        "architecture": "Model 4 Peak Capacity Hierarchical Attention CJPE"
    }
    with open(save_dir / "hierarchical_config.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    print(f"[CHECKPOINT] Saved checkpoint to {save_dir}")

print("[TRAIN] Starting Hierarchical training loop...")
global_step = 0
best_macro_f1 = 0.0
accum_steps = CFG["gradient_accumulation_steps"]
train_iter = iter(train_loader)
best_threshold_global = 0.5

model.train()
start_time = time.time()

while global_step < CFG["max_steps"]:
    optimizer.zero_grad()
    accum_loss = 0.0

    for _ in range(accum_steps):
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)

        chunk_ids = batch["chunk_ids"].to(device)
        chunk_masks = batch["chunk_masks"].to(device)
        doc_mask = batch["doc_chunk_mask"].to(device)
        labels = batch["labels"].to(device)

        logits, _ = model(chunk_ids, chunk_masks, doc_mask)
        loss = loss_fn(logits, labels) / accum_steps
        loss.backward()
        accum_loss += loss.item()

    torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
    optimizer.step()
    lr_scheduler.step()
    global_step += 1

    if global_step % 5 == 0 or global_step == 1:
        elapsed = time.time() - start_time
        print(f"Step {global_step:4d}/{CFG['max_steps']} | Smoothed BCE: {accum_loss:.4f} | LR: {lr_scheduler.get_last_lr()[0]:.2e} | Elapsed: {elapsed:.1f}s")

    # Periodic Validation & Sync
    if global_step % CFG["eval_interval"] == 0 or global_step == CFG["max_steps"]:
        macro_f1, acc, opt_th = evaluate_dev(model, dev_loader)
        print(f"--> [EVAL] Step {global_step} | Calibrated Macro-F1: {macro_f1:.4f} | Accuracy: {acc:.4f} | Optimal Threshold: {opt_th:.2f}")

        ckpt_dir = LOCAL_CHECKPOINT_DIR / f"checkpoint-{global_step}"
        save_hierarchical_checkpoint(ckpt_dir, global_step, opt_th)
        hub_sync.push_checkpoint_async(ckpt_dir, global_step)

        if macro_f1 > best_macro_f1:
            best_macro_f1 = macro_f1
            best_threshold_global = opt_th
            best_dir = LOCAL_CHECKPOINT_DIR / "checkpoint-best"
            save_hierarchical_checkpoint(best_dir, global_step, opt_th)
            hub_sync.push_checkpoint_async(best_dir, global_step)
            print(f"--> [BEST] New best Macro-F1: {best_macro_f1:.4f}!")

print("[TRAIN] Hierarchical training successfully completed!")


In [ ]:
# Cell 11: Crash Recovery Verification
print("[RECOVERY] Checking local and remote Hierarchical checkpoints...")
best_file = LOCAL_CHECKPOINT_DIR / "checkpoint-best" / "hierarchical_model.pt"
if best_file.exists():
    state_dict = torch.load(best_file, map_location=device)
    model.load_state_dict(state_dict)
    with open(LOCAL_CHECKPOINT_DIR / "checkpoint-best" / "hierarchical_config.json", "r") as f:
        meta = json.load(f)
        best_threshold_global = float(meta.get("best_threshold", 0.5))
    print(f"[RECOVERY] Successfully reloaded best Hierarchical model from {best_file} (threshold: {best_threshold_global:.2f})!")
else:
    print("[RECOVERY] Active Hierarchical weights in memory will be utilized for inference.")


In [ ]:
# Cell 12: Production Test Inference & Decision Thresholding
test_loader = DataLoader(
    test_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    collate_fn=chunk_collate_fn,
    drop_last=False,
    num_workers=0
)

print(f"[INFERENCE] Running test inference (decision threshold = {best_threshold_global:.2f})...")
model.eval()

test_predictions = []
start_gen = time.time()

with torch.no_grad():
    for batch in test_loader:
        chunk_ids = batch["chunk_ids"].to(device)
        chunk_masks = batch["chunk_masks"].to(device)
        doc_mask = batch["doc_chunk_mask"].to(device)
        doc_ids = batch["ids"]

        logits, _ = model(chunk_ids, chunk_masks, doc_mask)
        probs = torch.sigmoid(logits).cpu().numpy()

        for idx, doc_id in enumerate(doc_ids):
            pred_binary = int(probs[idx] >= best_threshold_global)
            test_predictions.append({
                "id": str(doc_id),
                "pred": pred_binary
            })

print(f"[INFERENCE] Inference complete across {len(test_predictions)} test cases in {time.time() - start_gen:.1f}s.")
ones = sum(1 for p in test_predictions if p["pred"] == 1)
zeros = sum(1 for p in test_predictions if p["pred"] == 0)
print(f"[DISTRIBUTION] Outcome Predictions: 0={zeros} ({zeros/len(test_predictions)*100:.1f}%), 1={ones} ({ones/len(test_predictions)*100:.1f}%)")


In [ ]:
# Cell 13: Submission Packaging & CodaBench Strict List Format Verification
import zipfile

# 1. Save prediction.json
pred_file = LOCAL_SUBMISSION_DIR / "prediction.json"
with open(pred_file, "w", encoding="utf-8") as f:
    json.dump(test_predictions, f, indent=2)

print(f"[SUBMISSION] Saved predictions to {pred_file}")

# 2. Strict CodaBench Verification
with open(pred_file, "r", encoding="utf-8") as f:
    loaded_preds = json.load(f)

assert isinstance(loaded_preds, list), f"Expected list format, got {type(loaded_preds)}"
assert len(loaded_preds) == len(test_records), f"Count mismatch: expected {len(test_records)}, got {len(loaded_preds)}"

for record in loaded_preds:
    assert "id" in record and "pred" in record, f"Missing key in {record}"
    assert isinstance(record["id"], str) and len(record["id"]) > 0, "Empty ID"
    assert record["pred"] in (0, 1), f"Invalid prediction value: {record['pred']}"

print(f"[VERIFIED] All {len(loaded_preds)} predictions strictly conform to CodaBench Phase 4 Format!")

# 3. Package root submission.zip
zip_path = LOCAL_OUTPUT_DIR / "submission.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_file, arcname="prediction.json")

print(f"[SUBMISSION] Packaged root submission zip at: {zip_path.resolve()} ({zip_path.stat().st_size / 1024:.2f} KB)")


In [ ]:
# Cell 14: Push Final Artifacts to Hugging Face Hub
print("[FINAL-SYNC] Pushing final evaluation artifacts to Hugging Face Hub...")
hub_sync.wait_all()

if HF_TOKEN and api:
    try:
        # Push prediction.json
        api.upload_file(
            path_or_fileobj=str(pred_file),
            repo_id=HUB_REPO_ID,
            path_in_repo=f"{HUB_SUBFOLDER}/prediction.json",
            commit_message="Add final test predictions for Phase 4 Sub-07 Hierarchical"
        )
        # Push submission.zip
        api.upload_file(
            path_or_fileobj=str(zip_path),
            repo_id=HUB_REPO_ID,
            path_in_repo=f"{HUB_SUBFOLDER}/submission.zip",
            commit_message="Add final submission zip for Phase 4 Sub-07 Hierarchical"
        )
        print(f"[FINAL-SYNC] All artifacts successfully uploaded to {HUB_REPO_ID}/{HUB_SUBFOLDER}!")
    except Exception as e:
        print(f"[FINAL-SYNC-WARNING] Upload error: {e}")
else:
    print("[FINAL-SYNC] Offline mode active: Skipping remote sync.")

print("=" * 80)
print("CS-781 Phase 4: Sub-07 Hierarchical Cross-Chunk Pipeline Completed Successfully!")
print("=" * 80)
